In [2]:
# !pip install pandas supabase openpyxl sqlalchemy psycopg2-binary

import pandas as pd
import numpy as np
import os
import re
from supabase import create_client, Client
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Βάλε εδώ τα δικά σου στοιχεία από το Supabase
SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD"       # Αντικατέστησε

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Σύνδεση με Supabase επιτυχής")

✅ Σύνδεση με Supabase επιτυχής


In [4]:
def read_air_old_format(filepath, year, location):
    """
    Διαβάζει αρχεία 2017-2021 με στήλες: time, spatial_ref, lon, lat, co, no, no2, so2, o3
    """
    df = pd.read_csv(filepath)
    
    # Κρατάμε μόνο τις χρήσιμες στήλες
    df = df[['time', 'co', 'no', 'no2', 'so2', 'o3']].copy()
    
    # Μετονομασία σε ομοιόμορφα ονόματα
    df.rename(columns={
        'co': 'co_conc',
        'no': 'no_conc', 
        'no2': 'no2_conc',
        'so2': 'so2_conc',
        'o3': 'o3_conc'
    }, inplace=True)
    
    df['timestamp'] = pd.to_datetime(df['time'])
    df['year'] = df['timestamp'].dt.year
    df['month'] = df['timestamp'].dt.month
    df['location'] = location  # 'kordelio-evosmos' ή ανάλογα
    df['data_source'] = 'old_format'
    
    # Αφαίρεση της αρχικής στήλης time
    df.drop('time', axis=1, inplace=True)
    
    return df

# Παράδειγμα χρήσης:
# df_2018 = read_air_old_format('path/to/2018_file.csv', 2018, 'kordelio-evosmos')

In [1]:
import requests
import socket

# Δοκιμή 1: Μπορείς να δεις το internet;
try:
    requests.get("https://www.google.com", timeout=5)
    print("✅ Internet λειτουργεί")
except:
    print("❌ Δεν υπάρχει internet")

# Δοκιμή 2: Μπορείς να δεις το Supabase;
url = "https://hjfbcknlnvgwsfqhfxwx.supabase.co" # ΒΑΛΕ ΤΟ ΔΙΚΟ ΣΟΥ URL
try:
    r = requests.get(url, timeout=10)
    print(f"✅ Supabase προσβάσιμο (status: {r.status_code})")
except Exception as e:
    print(f"❌ Δεν φτάνει στο Supabase: {e}")

# Δοκιμή 3: DNS lookup
hostname = url.replace("https://", "").replace("http://", "").split("/")[0]
try:
    socket.gethostbyname(hostname)
    print(f"✅ DNS lookup για {hostname} επιτυχές")
except:
    print(f"❌ DNS lookup απέτυχε για {hostname}")

✅ Internet λειτουργεί
❌ Δεν φτάνει στο Supabase: HTTPSConnectionPool(host='hjfbcknlnvgwsfqhfxwx.supabase.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='hjfbcknlnvgwsfqhfxwx.supabase.co', port=443): Failed to resolve 'hjfbcknlnvgwsfqhfxwx.supabase.co' ([Errno 11001] getaddrinfo failed)"))
❌ DNS lookup απέτυχε για hjfbcknlnvgwsfqhfxwx.supabase.co


In [1]:
from supabase import create_client

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD" 

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print("="*60)
print("ΕΛΕΓΧΟΣ ΔΕΔΟΜΕΝΩΝ ΣΤΟ SUPABASE")
print("="*60)

# 1. Τελευταίες εγγραφές για Κορδελιό
print("\n📌 Τελευταίες 5 εγγραφές για Kordelio:")
result = supabase.table('air_quality')\
    .select('date, no2, o3, co, so2, no')\
    .eq('location', 'Kordelio')\
    .order('date', desc=True)\
    .limit(5)\
    .execute()

for row in result.data:
    print(f"   {row['date']}: NO₂={row['no2']}, O₃={row['o3']}, CO={row['co']}, NO={row['no']}")

# 2. Έλεγχος αν υπάρχει ΚΑΘΟΛΟΥ NO για Κορδελιό
print("\n📌 Έλεγχος για NO > 0 σε Kordelio:")
no_check = supabase.table('air_quality')\
    .select('no')\
    .eq('location', 'Kordelio')\
    .gt('no', 0)\
    .limit(5)\
    .execute()

if no_check.data:
    print(f"   ✅ Υπάρχει NO: {len(no_check.data)} γραμμές")
    for row in no_check.data[:3]:
        print(f"      NO = {row['no']}")
else:
    print("   ❌ ΔΕΝ ΥΠΑΡΧΕΙ ΚΑΜΙΑ γραμμή με NO > 0 για Κορδελιό")

# 3. Πόσες γραμμές έχει συνολικά η Κορδελιό
total_kord = supabase.table('air_quality')\
    .select('*', count='exact')\
    .eq('location', 'Kordelio')\
    .execute()
print(f"\n📌 Σύνολο γραμμών για Kordelio: {total_kord.count}")

# 4. Έλεγχος για Evosmos
print("\n📌 Έλεγχος για Evosmos (Kordelio-Evosmos):")
result_evos = supabase.table('air_quality')\
    .select('date, no2, o3')\
    .eq('location', 'Kordelio-Evosmos')\
    .order('date', desc=True)\
    .limit(3)\
    .execute()

for row in result_evos.data:
    print(f"   {row['date']}: NO₂={row['no2']}, O₃={row['o3']}")

ΕΛΕΓΧΟΣ ΔΕΔΟΜΕΝΩΝ ΣΤΟ SUPABASE

📌 Τελευταίες 5 εγγραφές για Kordelio:
   2024-12-31: NO₂=0, O₃=0, CO=0, NO=0
   2024-12-31: NO₂=0, O₃=0, CO=0, NO=0
   2024-12-31: NO₂=0, O₃=0, CO=0, NO=0
   2024-12-31: NO₂=0, O₃=0, CO=0, NO=0
   2024-12-31: NO₂=0, O₃=0, CO=0, NO=0

📌 Έλεγχος για NO > 0 σε Kordelio:
   ❌ ΔΕΝ ΥΠΑΡΧΕΙ ΚΑΜΙΑ γραμμή με NO > 0 για Κορδελιό

📌 Σύνολο γραμμών για Kordelio: 26304

📌 Έλεγχος για Evosmos (Kordelio-Evosmos):
   2021-12-31: NO₂=0, O₃=0
   2021-12-31: NO₂=0, O₃=0
   2021-12-31: NO₂=0, O₃=0


In [2]:
from supabase import create_client

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD" 

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Διαγραφή όλων των λάθος δεδομένων (προσοχή!)
print("🗑️ Διαγραφή όλων των δεδομένων αέρα...")
supabase.table('air_quality').delete().neq('id', 0).execute()
print("✅ Διαγράφηκαν")

🗑️ Διαγραφή όλων των δεδομένων αέρα...
✅ Διαγράφηκαν


In [5]:
def read_air_new_format(filepath, year, location):
    """
    Διαβάζει αρχεία 2022-2024 με στήλες: time, level, spatial_ref, no2_conc, o3_conc, co_conc, no_conc, so2_conc
    """
    df = pd.read_csv(filepath)
    
    # Κρατάμε μόνο τις χρήσιμες στήλες
    df = df[['time', 'co_conc', 'no_conc', 'no2_conc', 'so2_conc', 'o3_conc']].copy()
    
    df['timestamp'] = pd.to_datetime(df['time'])
    df['year'] = df['timestamp'].dt.year
    df['month'] = df['timestamp'].dt.month
    df['location'] = location
    df['data_source'] = 'new_format'
    
    df.drop('time', axis=1, inplace=True)
    
    return df

# Παράδειγμα χρήσης:
# df_2024 = read_air_new_format('path/to/2024_file.csv', 2024, 'kordelio')

In [6]:
import pandas as pd
import glob
import os
from datetime import datetime

air_folder = 'data/air/'
all_files = glob.glob(os.path.join(air_folder, '*.csv'))

def read_air_file(filepath):
    """Διαβάζει σωστά και τις δύο μορφές CSV"""
    df = pd.read_csv(filepath)
    
    # Νέα μορφή (2022-2024): time, no2_conc, o3_conc, co_conc, no_conc, so2_conc
    if 'no2_conc' in df.columns:
        df = df.rename(columns={
            'no2_conc': 'no2',
            'o3_conc': 'o3',
            'co_conc': 'co',
            'no_conc': 'no',
            'so2_conc': 'so2'
        })
    
    # Παλιά μορφή (2017-2021): time, co, no, no2, so2, o3
    # ήδη έχει σωστά ονόματα
    
    # Κρατάμε μόνο τις απαραίτητες στήλες
    needed = ['time', 'co', 'no', 'no2', 'so2', 'o3']
    existing = [c for c in needed if c in df.columns]
    df = df[existing]
    
    # Προσθήκη γεωγραφικών αν δεν υπάρχουν (default)
    if 'lon' not in df.columns:
        df['lon'] = 22.9
    if 'lat' not in df.columns:
        df['lat'] = 40.66
    
    return df

all_data = []

for filepath in all_files:
    filename = os.path.basename(filepath)
    print(f"📖 Διαβάζω: {filename}")
    
    df = read_air_file(filepath)
    
    # Προσθήκη metadata
    df['timestamp'] = pd.to_datetime(df['time'])
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['year'] = df['timestamp'].dt.year
    
    # Προσθήκη location με βάση το όνομα αρχείου
    if 'kordeliou-euosmou' in filename.lower():
        df['location'] = 'Kordelio-Evosmos'
    elif 'kordeliou' in filename.lower():
        df['location'] = 'Kordelio'
    else:
        df['location'] = 'unknown'
    
    # Καθαρισμός: κρατάμε μόνο τις χρήσιμες στήλες
    df_clean = df[['timestamp', 'date', 'hour', 'year', 'location', 
                    'co', 'no', 'no2', 'so2', 'o3', 'lon', 'lat']].copy()
    
    # Αντικατάσταση NaN με 0
    df_clean = df_clean.fillna(0)
    
    all_data.append(df_clean)
    print(f"   ✅ {len(df_clean)} γραμμές")

# Ενοποίηση
air_combined = pd.concat(all_data, ignore_index=True)
print(f"\n📊 Σύνολο: {len(air_combined)} γραμμές")
print(f"   Τοποθεσίες: {air_combined['location'].unique()}")
print(f"   Έτη: {sorted(air_combined['year'].unique())}")

📖 Διαβάζω: municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2017.csv
   ✅ 78840 γραμμές
📖 Διαβάζω: municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2018.csv
   ✅ 78840 γραμμές
📖 Διαβάζω: municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2019.csv
   ✅ 131400 γραμμές
📖 Διαβάζω: municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2020.csv
   ✅ 193248 γραμμές
📖 Διαβάζω: municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2021.csv
   ✅ 192720 γραμμές
📖 Διαβάζω: municipality_of_kordeliou_pollutants_conc_timeseries-yearly_2022.csv
   ✅ 8760 γραμμές
📖 Διαβάζω: municipality_of_kordeliou_pollutants_conc_timeseries-yearly_2023.csv
   ✅ 8760 γραμμές
📖 Διαβάζω: municipality_of_kordeliou_pollutants_conc_timeseries-yearly_2024.csv
   ✅ 8784 γραμμές

📊 Σύνολο: 701352 γραμμές
   Τοποθεσίες: <StringArray>
['Kordelio-Evosmos', 'Kordelio']
Length: 2, dtype: str
   Έτη: [np.int32(2017), np.int32(2018), np.int32(2019), n

In [9]:
from supabase import create_client
import pandas as pd
from datetime import datetime, date

# ============================================
# ΣΤΟΙΧΕΙΑ SUPABASE (ΒΑΛΕ ΤΑ ΔΙΚΑ ΣΟΥ)
# ============================================
SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD" 
# ============================================

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Συνδέθηκε στο Supabase")

# ============================================
# 1. ΔΙΑΓΡΑΦΗ ΟΛΩΝ ΤΩΝ ΠΑΛΙΩΝ ΔΕΔΟΜΕΝΩΝ
# ============================================
print("\n🗑️ Διαγραφή όλων των παλιών δεδομένων αέρα...")
try:
    supabase.table('air_quality').delete().neq('id', 0).execute()
    print("   ✅ Διαγράφηκαν")
except Exception as e:
    print(f"   ⚠️ Σφάλμα διαγραφής: {e}")

# ============================================
# 2. ΠΡΟΕΤΟΙΜΑΣΙΑ ΔΕΔΟΜΕΝΩΝ ΓΙΑ UPLOAD
# ============================================
print("\n🔄 Προετοιμασία δεδομένων για upload...")

# Δημιουργία αντιγράφου
air_upload = air_combined.copy()

# Μετατροπή date και timestamp σε STRING
air_upload['date'] = air_upload['date'].astype(str)
air_upload['timestamp'] = air_upload['timestamp'].astype(str)

# Μετατροπή hour και year σε INTEGER (όχι float)
air_upload['hour'] = air_upload['hour'].fillna(0).astype(int)
air_upload['year'] = air_upload['year'].fillna(0).astype(int)

# Αντικατάσταση NaN και inf με 0
air_upload = air_upload.fillna(0)
air_upload = air_upload.replace([float('inf'), -float('inf')], 0)

# Βεβαιωθείτε ότι οι υπόλοιπες αριθμητικές στήλες είναι float
numeric_cols = ['co', 'no', 'no2', 'so2', 'o3', 'lon', 'lat']
for col in numeric_cols:
    if col in air_upload.columns:
        air_upload[col] = pd.to_numeric(air_upload[col], errors='coerce').fillna(0).astype(float)

print(f"   ✅ Έτοιμο: {len(air_upload)} γραμμές")
print(f"   📋 Τύποι δεδομένων:")
print(f"      hour: {air_upload['hour'].dtype}")
print(f"      year: {air_upload['year'].dtype}")
print(f"      no2: {air_upload['no2'].dtype}")

# ============================================
# 3. UPLOAD ΣΕ BATCHES
# ============================================
print(f"\n📤 Upload {len(air_upload)} γραμμών στη Supabase...")

batch_size = 500
total = len(air_upload)

for i in range(0, total, batch_size):
    batch = air_upload.iloc[i:i+batch_size].copy()
    
    # Καθαρισμός για JSON
    batch = batch.where(pd.notnull(batch), None)
    records = batch.to_dict(orient='records')
    
    # Τελικός καθαρισμός κάθε record
    clean_records = []
    for rec in records:
        clean_rec = {}
        for key, value in rec.items():
            if value is None or value == '':
                if key in ['hour', 'year']:
                    clean_rec[key] = 0
                else:
                    clean_rec[key] = 0.0
            elif isinstance(value, (datetime, date)):
                clean_rec[key] = str(value)
            elif isinstance(value, float) and (value != value):  # NaN check
                if key in ['hour', 'year']:
                    clean_rec[key] = 0
                else:
                    clean_rec[key] = 0.0
            elif key in ['hour', 'year']:
                # Εξασφάλιση ότι hour και year είναι integers
                clean_rec[key] = int(float(value))
            else:
                clean_rec[key] = value
        clean_records.append(clean_rec)
    
    try:
        supabase.table('air_quality').insert(clean_records).execute()
        print(f"   ✅ Ανέβηκαν {min(i+batch_size, total)}/{total}")
    except Exception as e:
        print(f"   ❌ Σφάλμα στο batch {i}: {e}")
        
        # Δοκιμή με μία μόνο γραμμή
        if i == 0:
            print("\n   🔍 Δοκιμή με μία γραμμή...")
            test_record = clean_records[0].copy()
            # Εξασφάλιση ότι hour και year είναι int
            test_record['hour'] = int(test_record['hour'])
            test_record['year'] = int(test_record['year'])
            try:
                supabase.table('air_quality').insert(test_record).execute()
                print("   ✅ Η μία γραμμή ανέβηκε!")
            except Exception as e2:
                print(f"   ❌ Ούτε μία γραμμή: {e2}")
                print(f"   Record: {test_record}")
        break

print("\n✅ Ολοκληρώθηκε το upload!")

✅ Συνδέθηκε στο Supabase

🗑️ Διαγραφή όλων των παλιών δεδομένων αέρα...
   ✅ Διαγράφηκαν

🔄 Προετοιμασία δεδομένων για upload...
   ✅ Έτοιμο: 701352 γραμμές
   📋 Τύποι δεδομένων:
      hour: int64
      year: int64
      no2: float64

📤 Upload 701352 γραμμών στη Supabase...
   ✅ Ανέβηκαν 500/701352
   ✅ Ανέβηκαν 1000/701352
   ✅ Ανέβηκαν 1500/701352
   ✅ Ανέβηκαν 2000/701352
   ✅ Ανέβηκαν 2500/701352
   ✅ Ανέβηκαν 3000/701352
   ✅ Ανέβηκαν 3500/701352
   ✅ Ανέβηκαν 4000/701352
   ✅ Ανέβηκαν 4500/701352
   ✅ Ανέβηκαν 5000/701352
   ✅ Ανέβηκαν 5500/701352
   ✅ Ανέβηκαν 6000/701352
   ✅ Ανέβηκαν 6500/701352
   ✅ Ανέβηκαν 7000/701352
   ✅ Ανέβηκαν 7500/701352
   ✅ Ανέβηκαν 8000/701352
   ✅ Ανέβηκαν 8500/701352
   ✅ Ανέβηκαν 9000/701352
   ✅ Ανέβηκαν 9500/701352
   ✅ Ανέβηκαν 10000/701352
   ✅ Ανέβηκαν 10500/701352
   ✅ Ανέβηκαν 11000/701352
   ✅ Ανέβηκαν 11500/701352
   ✅ Ανέβηκαν 12000/701352
   ✅ Ανέβηκαν 12500/701352
   ✅ Ανέβηκαν 13000/701352
   ✅ Ανέβηκαν 13500/701352
   ✅ Ανέβηκαν 14

In [4]:
import os
print("Τρέχων φάκελος:", os.getcwd())
print("Υπάρχει data/air;", os.path.exists('data/air'))
print("Αρχεία στον τρέχοντα φάκελο:", os.listdir('.'))

Τρέχων φάκελος: c:\Users\User\Desktop\air_water_quality\notebooks
Υπάρχει data/air; False
Αρχεία στον τρέχοντα φάκελο: ['process_data.ipynb']


In [10]:
from supabase import create_client

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD" 

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Έλεγχος για SO2 και NO στον Εύοσμο
print("📌 ΕΛΕΓΧΟΣ ΓΙΑ ΕΥΟΣΜΟ (Kordelio-Evosmos)")
print("="*50)

for pollutant in ['so2', 'no']:
    result = supabase.table('air_quality')\
        .select(pollutant)\
        .eq('location', 'Kordelio-Evosmos')\
        .gt(pollutant, 0)\
        .limit(5)\
        .execute()
    
    print(f"{pollutant.upper()} > 0: {len(result.data)} γραμμές")
    if result.data:
        print(f"   Παράδειγμα: {result.data[0][pollutant]}")

# Έλεγχος για τα έτη που έχει δεδομένα ο Εύοσμος
print("\n📌 ΕΤΗ ΜΕ ΔΕΔΟΜΕΝΑ ΓΙΑ ΕΥΟΣΜΟ:")
years = supabase.table('air_quality')\
    .select('year')\
    .eq('location', 'Kordelio-Evosmos')\
    .gt('no2', 0)\
    .execute()

unique_years = sorted(set([row['year'] for row in years.data]))
print(f"   {unique_years}")

📌 ΕΛΕΓΧΟΣ ΓΙΑ ΕΥΟΣΜΟ (Kordelio-Evosmos)
SO2 > 0: 5 γραμμές
   Παράδειγμα: 3.4482422
NO > 0: 5 γραμμές
   Παράδειγμα: 0.10723877

📌 ΕΤΗ ΜΕ ΔΕΔΟΜΕΝΑ ΓΙΑ ΕΥΟΣΜΟ:
   [2019]


In [11]:
from supabase import create_client

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD" 

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Έλεγχος για NO στο Κορδελιό
print("📌 NO για Κορδελιό:")
result = supabase.table('air_quality')\
    .select('date, no')\
    .eq('location', 'Kordelio')\
    .gt('no', 0)\
    .order('date', desc=True)\
    .limit(5)\
    .execute()

if result.data:
    for row in result.data:
        print(f"   {row['date']}: NO = {row['no']}")
else:
    print("   ❌ ΚΑΜΙΑ γραμμή με NO > 0")

# Έλεγχος για NO στον Εύοσμο
print("\n📌 NO για Εύοσμο (Kordelio-Evosmos):")
result2 = supabase.table('air_quality')\
    .select('date, no')\
    .eq('location', 'Kordelio-Evosmos')\
    .gt('no', 0)\
    .order('date', desc=True)\
    .limit(5)\
    .execute()

if result2.data:
    for row in result2.data:
        print(f"   {row['date']}: NO = {row['no']}")
else:
    print("   ❌ ΚΑΜΙΑ γραμμή με NO > 0")

📌 NO για Κορδελιό:
   2024-12-31: NO = 0.06768453
   2024-12-31: NO = 0.070828676
   2024-12-31: NO = 0.08778755
   2024-12-31: NO = 0.07546674
   2024-12-31: NO = 0.15085089

📌 NO για Εύοσμο (Kordelio-Evosmos):
   2021-12-31: NO = 0.2923584
   2021-12-31: NO = 0.07589722
   2021-12-31: NO = 0.056655884
   2021-12-31: NO = 0.17285156
   2021-12-31: NO = 0.105285645


In [12]:
from supabase import create_client

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD" 

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Δες όλες τις διαφορετικές τιμές location που υπάρχουν
result = supabase.table('air_quality').select('location').execute()
locations = set([row['location'] for row in result.data])
print("📍 Όλες οι location στη βάση:")
for loc in locations:
    print(f"   '{loc}'")

# Έλεγχος για τον Εύοσμο με κάθε πιθανή ονομασία
for loc in locations:
    if 'evos' in loc.lower() or 'euosm' in loc.lower():
        count = supabase.table('air_quality').select('*', count='exact').eq('location', loc).execute()
        print(f"\n{loc}: {count.count} γραμμές")
        
        # Έλεγχος για NO > 0
        no_check = supabase.table('air_quality').select('no').eq('location', loc).gt('no', 0).limit(3).execute()
        print(f"   NO > 0: {len(no_check.data)} γραμμές")

📍 Όλες οι location στη βάση:
   'Kordelio-Evosmos'

Kordelio-Evosmos: 675048 γραμμές
   NO > 0: 3 γραμμές


In [5]:
import os

# Πήγαινε στον γονικό φάκελο (εκεί που είναι το data/)
os.chdir('..')
print("Νέος φάκελος:", os.getcwd())
print("Υπάρχει data/air;", os.path.exists('data/air'))

Νέος φάκελος: c:\Users\User\Desktop\air_water_quality
Υπάρχει data/air; True


In [12]:
import os
print("Τρέχων φάκελος:", os.getcwd())
print("Υπάρχει data/air;", os.path.exists('data/air'))
print("Περιεχόμενα τρέχοντος φακέλου:", os.listdir('.'))

Τρέχων φάκελος: c:\Users\User\Desktop\air_water_quality\notebooks
Υπάρχει data/air; False
Περιεχόμενα τρέχοντος φακέλου: ['process_data.ipynb']


In [13]:
import os

# Πας ένα επίπεδο πάνω (στον κεντρικό φάκελο)
os.chdir('..')

print("Νέος τρέχων φάκελος:", os.getcwd())
print("Υπάρχει data/air;", os.path.exists('data/air'))
print("Υπάρχει data/water;", os.path.exists('data/water'))

Νέος τρέχων φάκελος: c:\Users\User\Desktop\air_water_quality
Υπάρχει data/air; True
Υπάρχει data/water; True


In [16]:
# ============================================
# ΚΕΛΙ 5 – ΦΟΡΤΩΣΗ ΟΛΩΝ ΤΩΝ ΑΡΧΕΙΩΝ ΑΕΡΑ (ΑΥΤΟΜΑΤΑ)
# ============================================

import os
import pandas as pd
import glob
import re

print("📁 Αναζήτηση αρχείων αέρα...")

# Φάκελος με τα CSV
air_folder = 'data/air/'

# Βρες όλα τα CSV αρχεία
all_csv_files = glob.glob(os.path.join(air_folder, '*.csv'))
all_csv_files += glob.glob(os.path.join(air_folder, '*.CSV'))

print(f"\n📊 Βρέθηκαν {len(all_csv_files)} αρχεία CSV:")
for f in all_csv_files:
    print(f"   - {os.path.basename(f)}")

if len(all_csv_files) == 0:
    print("\n❌ ΔΕΝ βρέθηκαν CSV. Έλεγξε τον φάκελο data/air/")
else:
    all_air_dfs = []
    
    for filepath in all_csv_files:
        filename = os.path.basename(filepath)
        print(f"\n📖 Διαβάζω: {filename}")
        
        try:
            df = pd.read_csv(filepath)
            print(f"   ✅ {len(df)} γραμμές, {len(df.columns)} στήλες")
            print(f"   Στήλες: {', '.join(df.columns[:6])}...")
            
            # Προσθήκη πληροφορίας προέλευσης
            df['source_file'] = filename
            
            # Εξαγωγή έτους από το όνομα αρχείου (π.χ. ..._2024.csv)
            year_match = re.search(r'(\d{4})', filename)
            if year_match:
                df['year'] = int(year_match.group(1))
            else:
                df['year'] = None
            
            # Εξαγωγή τοποθεσίας από το όνομα
            if 'kordeliou-euosmou' in filename.lower():
                df['location'] = 'kordelio-evosmos'
            elif 'kordeliou' in filename.lower() and 'euosmou' not in filename.lower():
                df['location'] = 'kordelio'
            else:
                df['location'] = 'unknown'
            
            all_air_dfs.append(df)
            
        except Exception as e:
            print(f"   ❌ Σφάλμα: {e}")
    
    # Ενοποίηση όλων των DataFrames
    if all_air_dfs:
        air_combined = pd.concat(all_air_dfs, ignore_index=True)
        
        print("\n" + "="*60)
        print(f"📊 ΣΥΝΟΛΟ ΔΕΔΟΜΕΝΩΝ ΑΕΡΑ")
        print("="*60)
        print(f"✅ Σύνολο γραμμών: {len(air_combined)}")
        print(f"✅ Σύνολο στηλών: {len(air_combined.columns)}")
        print(f"\n📋 Ονόματα στηλών:")
        for col in air_combined.columns:
            print(f"   - {col}")
        
        print(f"\n📅 Έτη που καλύπτονται: {sorted(air_combined['year'].dropna().unique())}")
        print(f"📍 Τοποθεσίες: {air_combined['location'].unique()}")
        
        print(f"\n🔍 Πρώτες 3 γραμμές:")
        print(air_combined.head(3))
        
        # Αποθήκευση για χρήση στα επόμενα κελιά
        %store air_combined
        
    else:
        print("\n❌ Κανένα αρχείο δεν διαβάστηκε επιτυχώς")

📁 Αναζήτηση αρχείων αέρα...

📊 Βρέθηκαν 16 αρχεία CSV:
   - municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2017.csv
   - municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2018.csv
   - municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2019.csv
   - municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2020.csv
   - municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2021.csv
   - municipality_of_kordeliou_pollutants_conc_timeseries-yearly_2022.csv
   - municipality_of_kordeliou_pollutants_conc_timeseries-yearly_2023.csv
   - municipality_of_kordeliou_pollutants_conc_timeseries-yearly_2024.csv
   - municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2017.csv
   - municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2018.csv
   - municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-yearly_2019.csv
   - municipality_of_kordeliou-euosmou_pollutants_conc_timeseries-y

In [17]:
# ============================================
# ΚΕΛΙ 5β – ΚΑΘΑΡΙΣΜΟΣ ΔΕΔΟΜΕΝΩΝ ΑΕΡΑ
# ============================================

print("🔄 Καθαρισμός δεδομένων αέρα...")

# Αφαίρεση διπλότυπων αρχείων (αν υπάρχουν ίδιες γραμμές)
air_combined = air_combined.drop_duplicates(subset=['time'], keep='first')

print(f"✅ Μετά από αφαίρεση διπλότυπων: {len(air_combined)} γραμμές")

# Καθαρισμός τιμών – αντικατάσταση NaN με 0 ή None
numeric_columns = ['co', 'no', 'no2', 'so2', 'o3', 'co_conc', 'no_conc', 'no2_conc', 'so2_conc', 'o3_conc']

for col in numeric_columns:
    if col in air_combined.columns:
        # Αντικατάσταση NaN με 0
        air_combined[col] = pd.to_numeric(air_combined[col], errors='coerce').fillna(0)

# Δημιουργία στήλης timestamp
if 'time' in air_combined.columns:
    air_combined['timestamp'] = pd.to_datetime(air_combined['time'])
    air_combined['date'] = air_combined['timestamp'].dt.date
    air_combined['hour'] = air_combined['timestamp'].dt.hour

# Ενοποίηση ονομάτων ρύπων (για να έχουμε ίδιες στήλες σε όλα τα αρχεία)
# Αν υπάρχει co_conc, μετονόμασε σε co
if 'co_conc' in air_combined.columns and 'co' not in air_combined.columns:
    air_combined['co'] = air_combined['co_conc']
if 'no_conc' in air_combined.columns and 'no' not in air_combined.columns:
    air_combined['no'] = air_combined['no_conc']
if 'no2_conc' in air_combined.columns and 'no2' not in air_combined.columns:
    air_combined['no2'] = air_combined['no2_conc']
if 'so2_conc' in air_combined.columns and 'so2' not in air_combined.columns:
    air_combined['so2'] = air_combined['so2_conc']
if 'o3_conc' in air_combined.columns and 'o3' not in air_combined.columns:
    air_combined['o3'] = air_combined['o3_conc']

# Κράτα μόνο τις χρήσιμες στήλες
keep_columns = ['timestamp', 'date', 'hour', 'year', 'location', 'co', 'no', 'no2', 'so2', 'o3', 'lon', 'lat']
existing_columns = [col for col in keep_columns if col in air_combined.columns]
air_clean = air_combined[existing_columns].copy()

print(f"\n📊 Στατιστικά καθαρισμένων δεδομένων:")
print(f"   - Γραμμές: {len(air_clean)}")
print(f"   - Στήλες: {air_clean.columns.tolist()}")
print(f"   - Έτη: {sorted(air_clean['year'].dropna().unique())}")
print(f"   - Τοποθεσίες: {air_clean['location'].unique()}")

print(f"\n🔍 Δείγμα καθαρισμένων δεδομένων:")
print(air_clean.head(5))

# Αποθήκευση τοπικά
air_clean.to_csv('data/air_quality_clean.csv', index=False)
print("\n💾 Αποθηκεύτηκε σε: data/air_quality_clean.csv")

%store air_clean

🔄 Καθαρισμός δεδομένων αέρα...
✅ Μετά από αφαίρεση διπλότυπων: 70128 γραμμές

📊 Στατιστικά καθαρισμένων δεδομένων:
   - Γραμμές: 70128
   - Στήλες: ['timestamp', 'date', 'hour', 'year', 'location', 'co', 'no', 'no2', 'so2', 'o3', 'lon', 'lat']
   - Έτη: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   - Τοποθεσίες: <StringArray>
['kordelio-evosmos', 'kordelio']
Length: 2, dtype: str

🔍 Δείγμα καθαρισμένων δεδομένων:
             timestamp        date  hour  year          location         co  \
0  2017-01-01 00:00:00  2017-01-01     0  2017  kordelio-evosmos  281.36383   
9  2017-01-01 01:00:00  2017-01-01     1  2017  kordelio-evosmos  264.59735   
18 2017-01-01 02:00:00  2017-01-01     2  2017  kordelio-evosmos  257.38742   
27 2017-01-01 03:00:00  2017-01-01     3  2017  kordelio-evosmos  262.89825   
36 2017-01-01 04:00:00  2017-01-01     4  2017  kordelio-evosmos  270.64690   

     no        no2    

In [18]:
# ============================================
# ΚΕΛΙ 6 – ΦΟΡΤΩΣΗ ΟΛΩΝ ΤΩΝ ΑΡΧΕΙΩΝ ΝΕΡΟΥ
# ============================================

import pandas as pd
import glob
import os

print("📁 Αναζήτηση αρχείων νερού...")

water_folder = 'data/water/'

# Βρες όλα τα Excel αρχεία
all_excel_files = glob.glob(os.path.join(water_folder, '*.xlsx'))
all_excel_files += glob.glob(os.path.join(water_folder, '*.XLSX'))

# Αφαίρεση διπλότυπων ονομάτων
all_excel_files = list(set(all_excel_files))

print(f"\n📊 Βρέθηκαν {len(all_excel_files)} αρχεία Excel:")
for f in all_excel_files:
    print(f"   - {os.path.basename(f)}")

if len(all_excel_files) == 0:
    print("\n❌ ΔΕΝ βρέθηκαν Excel αρχεία στον φάκελο data/water/")
else:
    all_water_dfs = []
    
    for filepath in all_excel_files:
        filename = os.path.basename(filepath)
        location = filename.replace('.xlsx', '').replace('.XLSX', '')
        
        print(f"\n📖 Διαβάζω: {filename}")
        
        try:
            # Διάβασε το πρώτο sheet
            df = pd.read_excel(filepath, sheet_name=0)
            print(f"   ✅ {len(df)} γραμμές, {len(df.columns)} στήλες")
            print(f"   Στήλες: {df.columns.tolist()}")
            
            # Προσθήκη στήλης τοποθεσίας
            df['location'] = location
            
            all_water_dfs.append(df)
            
        except Exception as e:
            print(f"   ❌ Σφάλμα: {e}")
    
    if all_water_dfs:
        water_combined = pd.concat(all_water_dfs, ignore_index=True)
        
        print("\n" + "="*60)
        print(f"📊 ΣΥΝΟΛΟ ΔΕΔΟΜΕΝΩΝ ΝΕΡΟΥ")
        print("="*60)
        print(f"✅ Σύνολο γραμμών: {len(water_combined)}")
        print(f"✅ Στήλες: {water_combined.columns.tolist()}")
        
        print(f"\n🔍 Πρώτες 3 γραμμές:")
        print(water_combined.head(3))
        
        # Αποθήκευση τοπικά
        water_combined.to_csv('data/water_quality_clean.csv', index=False)
        print("\n💾 Αποθηκεύτηκε σε: data/water_quality_clean.csv")
        
        %store water_combined
        
    else:
        print("\n❌ Κανένα αρχείο νερού δεν διαβάστηκε")

📁 Αναζήτηση αρχείων νερού...

📊 Βρέθηκαν 3 αρχεία Excel:
   - eyosmos.xlsx
   - kordelio.xlsx
   - dialogi.xlsx

📖 Διαβάζω: eyosmos.xlsx
   ✅ 28 γραμμές, 7 στήλες
   Στήλες: ['Year', 'Month', 'Category', 'Φυσικοχημικές Παράμετροι', 'Μονάδα μέτρησης', 'Τιμή', 'Παραμετρική τιμή1']

📖 Διαβάζω: kordelio.xlsx
   ✅ 28 γραμμές, 7 στήλες
   Στήλες: ['Year', 'Month', 'Category', 'Φυσικοχημικές Παράμετροι', 'Μονάδα μέτρησης', 'Τιμή', 'Παραμετρική τιμή1']

📖 Διαβάζω: dialogi.xlsx
   ✅ 28 γραμμές, 7 στήλες
   Στήλες: ['Year', 'Month', 'Category', 'Φυσικοχημικές Παράμετροι', 'Μονάδα μέτρησης', 'Τιμή', 'Παραμετρική τιμή1']

📊 ΣΥΝΟΛΟ ΔΕΔΟΜΕΝΩΝ ΝΕΡΟΥ
✅ Σύνολο γραμμών: 84
✅ Στήλες: ['Year', 'Month', 'Category', 'Φυσικοχημικές Παράμετροι', 'Μονάδα μέτρησης', 'Τιμή', 'Παραμετρική τιμή1', 'location']

🔍 Πρώτες 3 γραμμές:
   Year       Month Category Φυσικοχημικές Παράμετροι Μονάδα μέτρησης  \
0  2023  Δεκέμβριος  Εύοσμος             Θολότητα NTU             NTU   
1  2023  Δεκέμβριος  Εύοσμος             

In [21]:
# ============================================
# ΚΕΛΙ 7 – ΠΡΟΕΤΟΙΜΑΣΙΑ ΔΕΔΟΜΕΝΩΝ ΓΙΑ SUPABASE
# ============================================

import pandas as pd
import numpy as np

print("🔄 Προετοιμασία δεδομένων για Supabase...")

# ============================================
# 1. ΚΑΘΑΡΙΣΜΟΣ ΔΕΔΟΜΕΝΩΝ ΝΕΡΟΥ
# ============================================

# Φόρτωση των δεδομένων νερού από το Κελί 6
# (αν δεν υπάρχει το water_combined, το φορτώνουμε από το αρχείο)
try:
    water_combined
except NameError:
    water_combined = pd.read_csv('data/water_quality_clean.csv')
    print("✅ Φορτώθηκε water_combined από αρχείο")

water_clean = water_combined.copy()

# Καθαρισμός τιμών – αφαίρεση ^3, ^5 και μετατροπή < τιμών
def clean_value(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    # Αφαίρεση εκθετών
    val_str = val_str.replace('^3', '').replace('^5', '').replace('^2', '')
    # Αντικατάσταση κόμματος με τελεία
    val_str = val_str.replace(',', '.')
    # Αν έχει <, κρατάμε ως string
    if '<' in val_str:
        return val_str
    # Αν είναι ΔΠ (Δεν Προσδιορίστηκε)
    if val_str == 'ΔΠ':
        return None
    # Προσπάθεια μετατροπής σε αριθμό
    try:
        return float(val_str)
    except:
        return val_str

water_clean['clean_value'] = water_clean['Τιμή'].apply(clean_value)

# Μετονομασία στηλών για Supabase
water_clean = water_clean.rename(columns={
    'Year': 'year',
    'Month': 'month',
    'Category': 'category',
    'Φυσικοχημικές Παράμετροι': 'parameter',
    'Μονάδα μέτρησης': 'unit',
    'Παραμετρική τιμή1': 'limit_value',
    'location': 'location'
})

# Κράτα μόνο χρήσιμες στήλες
water_final = water_clean[['year', 'month', 'location', 'category', 'parameter', 'unit', 'clean_value', 'limit_value']].copy()

print(f"\n✅ Νερό: {len(water_final)} γραμμές")
print(water_final.head(5))

# ============================================
# 2. ΠΡΟΕΤΟΙΜΑΣΙΑ ΔΕΔΟΜΕΝΩΝ ΑΕΡΑ
# ============================================

try:
    air_clean
except NameError:
    air_clean = pd.read_csv('data/air_quality_clean.csv')
    print("✅ Φορτώθηκε air_clean από αρχείο")

air_final = air_clean.copy()

# Μετονομασία location για συνέπεια
air_final['location'] = air_final['location'].replace({
    'kordelio-evosmos': 'Kordelio-Evosmos',
    'kordelio': 'Kordelio'
})

# Επιλογή στηλών για Supabase
air_final = air_final[['timestamp', 'date', 'hour', 'year', 'location', 'co', 'no', 'no2', 'so2', 'o3', 'lon', 'lat']].copy()

print(f"\n✅ Αέρας: {len(air_final)} γραμμές")
print(air_final.head(5))

# ============================================
# 3. ΑΠΟΘΗΚΕΥΣΗ ΤΟΠΙΚΑ ΓΙΑ BACKUP
# ============================================

water_final.to_csv('data/water_final.csv', index=False, encoding='utf-8')
air_final.to_csv('data/air_final.csv', index=False, encoding='utf-8')

print("\n💾 Αποθηκεύτηκαν τελικά αρχεία:")
print("   - data/water_final.csv")
print("   - data/air_final.csv")

# Αποθήκευση στη μνήμη για τα επόμενα κελιά
%store water_final
%store air_final

print("\n✅ Έτοιμο για αποστολή στο Supabase (Κελί 8)")

🔄 Προετοιμασία δεδομένων για Supabase...

✅ Νερό: 84 γραμμές
   year       month location category     parameter              unit  \
0  2023  Δεκέμβριος  eyosmos  Εύοσμος  Θολότητα NTU               NTU   
1  2023  Δεκέμβριος  eyosmos  Εύοσμος         Χρώμα        mg/l Pt-Co   
2  2023  Δεκέμβριος  eyosmos  Εύοσμος       Αργίλιο           μg Al/l   
3  2023  Δεκέμβριος  eyosmos  Εύοσμος     Χλωριούχα        mg Cl^-^/l   
4  2023  Δεκέμβριος  eyosmos  Εύοσμος   Αγωγιμότητα  μS/cm στους 20°C   

  clean_value                                        limit_value  
0      <0.050                                               <1,0  
1         <5   Αποδεκτή στους καταναλωτές και άνευ ασυνήθους ...  
2         <25                                                200  
3        None                                                250  
4       367.0                                               2500  

✅ Αέρας: 70128 γραμμές
             timestamp        date  hour  year          location         c

In [23]:
# ============================================
# ΚΕΛΙ 8 – ΑΠΟΣΤΟΛΗ ΣΤΟ SUPABASE (ΔΙΟΡΘΩΜΕΝΟ)
# ============================================

from supabase import create_client
import pandas as pd
import time
from datetime import date

# ============================================
# 🔴 ΑΝΤΙΚΑΤΑΣΤΗΣΕ ΜΕ ΤΑ ΔΙΚΑ ΣΟΥ ΣΤΟΙΧΕΙΑ 🔴
# ============================================
SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD" 
# ============================================

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Συνδέθηκε στο Supabase")

# ============================================
# 1. ΑΠΟΣΤΟΛΗ ΔΕΔΟΜΕΝΩΝ ΝΕΡΟΥ (λειτουργεί ήδη)
# ============================================
print(f"\n📤 Αποστολή {len(water_final)} γραμμών νερού...")

water_batch_size = 20
water_success = 0

for i in range(0, len(water_final), water_batch_size):
    batch = water_final.iloc[i:i+water_batch_size].copy()
    batch = batch.where(pd.notnull(batch), None)
    records = batch.to_dict(orient='records')
    
    try:
        supabase.table('water_quality').insert(records).execute()
        water_success += len(records)
        print(f"   ✅ Ανέβηκαν {water_success}/{len(water_final)}")
        time.sleep(0.1)
    except Exception as e:
        print(f"   ❌ Σφάλμα: {e}")
        break

# ============================================
# 2. ΑΠΟΣΤΟΛΗ ΔΕΔΟΜΕΝΩΝ ΑΕΡΑ (ΔΙΟΡΘΩΜΕΝΟ)
# ============================================
print(f"\n📤 Αποστολή {len(air_final)} γραμμών αέρα...")

air_batch_size = 100
air_success = 0

for i in range(0, len(air_final), air_batch_size):
    batch = air_final.iloc[i:i+air_batch_size].copy()
    
    # Καθαρισμός για JSON serialization
    batch = batch.where(pd.notnull(batch), None)
    
    # ΜΕΤΑΤΡΟΠΗ date OBJECT ΣΕ STRING
    if 'date' in batch.columns:
        batch['date'] = batch['date'].astype(str)
    
    if 'timestamp' in batch.columns:
        batch['timestamp'] = batch['timestamp'].astype(str)
    
    # Μετατροπή float NaN σε None
    for col in ['co', 'no', 'no2', 'so2', 'o3', 'lon', 'lat']:
        if col in batch.columns:
            batch[col] = batch[col].where(pd.notnull(batch[col]), None)
    
    records = batch.to_dict(orient='records')
    
    try:
        supabase.table('air_quality').insert(records).execute()
        air_success += len(records)
        print(f"   ✅ Ανέβηκαν {air_success}/{len(air_final)}")
        time.sleep(0.05)
    except Exception as e:
        print(f"   ❌ Σφάλμα στο batch {i}: {e}")
        # Δοκίμασε με ένα μόνο record για debugging
        if i == 0:
            print("\n🔍 Δοκιμή με ένα μόνο record...")
            try:
                single_record = records[0]
                supabase.table('air_quality').insert(single_record).execute()
                print("   ✅ Το single record ανέβηκε!")
            except Exception as e2:
                print(f"   ❌ Σφάλμα και στο single: {e2}")
                print(f"   Record: {single_record}")
        break

# ============================================
# 3. ΕΛΕΓΧΟΣ
# ============================================
print("\n🔍 Έλεγχος Supabase:")

try:
    water_check = supabase.table('water_quality').select('*', count='exact').limit(3).execute()
    air_check = supabase.table('air_quality').select('*', count='exact').limit(3).execute()
    
    print(f"   ✅ Νερό: {water_check.count} γραμμές συνολικά")
    print(f"   ✅ Αέρας: {air_check.count} γραμμές συνολικά")
except Exception as e:
    print(f"   ❌ Σφάλμα ελέγχου: {e}")

✅ Συνδέθηκε στο Supabase

📤 Αποστολή 84 γραμμών νερού...
   ✅ Ανέβηκαν 20/84
   ✅ Ανέβηκαν 40/84
   ✅ Ανέβηκαν 60/84
   ✅ Ανέβηκαν 80/84
   ✅ Ανέβηκαν 84/84

📤 Αποστολή 70128 γραμμών αέρα...
   ✅ Ανέβηκαν 100/70128
   ✅ Ανέβηκαν 200/70128
   ✅ Ανέβηκαν 300/70128
   ✅ Ανέβηκαν 400/70128
   ✅ Ανέβηκαν 500/70128
   ✅ Ανέβηκαν 600/70128
   ✅ Ανέβηκαν 700/70128
   ✅ Ανέβηκαν 800/70128
   ✅ Ανέβηκαν 900/70128
   ✅ Ανέβηκαν 1000/70128
   ✅ Ανέβηκαν 1100/70128
   ✅ Ανέβηκαν 1200/70128
   ✅ Ανέβηκαν 1300/70128
   ✅ Ανέβηκαν 1400/70128
   ✅ Ανέβηκαν 1500/70128
   ✅ Ανέβηκαν 1600/70128
   ✅ Ανέβηκαν 1700/70128
   ✅ Ανέβηκαν 1800/70128
   ✅ Ανέβηκαν 1900/70128
   ✅ Ανέβηκαν 2000/70128
   ✅ Ανέβηκαν 2100/70128
   ✅ Ανέβηκαν 2200/70128
   ✅ Ανέβηκαν 2300/70128
   ✅ Ανέβηκαν 2400/70128
   ✅ Ανέβηκαν 2500/70128
   ✅ Ανέβηκαν 2600/70128
   ✅ Ανέβηκαν 2700/70128
   ✅ Ανέβηκαν 2800/70128
   ✅ Ανέβηκαν 2900/70128
   ✅ Ανέβηκαν 3000/70128
   ✅ Ανέβηκαν 3100/70128
   ✅ Ανέβηκαν 3200/70128
   ✅ Ανέβηκαν 3300

In [24]:
# ============================================
# ΚΕΛΙ 9 – ΚΑΘΑΡΙΣΜΟΣ ΚΑΙ ΣΥΝΕΧΙΣΗ UPLOAD
# ============================================

import pandas as pd
import numpy as np
from supabase import create_client
import time

# Στοιχεία σύνδεσης
SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Συνδέθηκε στο Supabase")

# ============================================
# 1. ΚΑΘΑΡΙΣΜΟΣ air_final (αφαίρεση inf, -inf, NaN)
# ============================================

print("\n🔄 Καθαρισμός δεδομένων αέρα...")

# Φόρτωση των αρχικών δεδομένων
air_final = pd.read_csv('data/air_final.csv')

# Αριθμός γραμμών πριν
print(f"   Πριν: {len(air_final)} γραμμές")

# Αντικατάσταση inf, -inf με 0
air_final = air_final.replace([np.inf, -np.inf], 0)

# Αντικατάσταση NaN με 0 (για αριθμητικές στήλες)
numeric_cols = ['co', 'no', 'no2', 'so2', 'o3', 'lon', 'lat', 'hour', 'year']
for col in numeric_cols:
    if col in air_final.columns:
        air_final[col] = pd.to_numeric(air_final[col], errors='coerce').fillna(0)

# Μετατροπή date/timestamp σε string
air_final['date'] = pd.to_datetime(air_final['date']).dt.strftime('%Y-%m-%d')
air_final['timestamp'] = pd.to_datetime(air_final['timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S')

print(f"   Μετά: {len(air_final)} γραμμές")

# ============================================
# 2. ΕΛΕΓΧΟΣ ΠΟΙΕΣ ΓΡΑΜΜΕΣ ΕΧΟΥΝ ΗΔΗ ΑΝΕΒΕΙ
# ============================================

print("\n🔍 Έλεγχος ήδη ανεβασμένων γραμμών...")

try:
    existing = supabase.table('air_quality').select('*', count='exact').execute()
    existing_count = existing.count
    print(f"   Ήδη ανεβασμένες: {existing_count} γραμμές")
    
    # Ανέβασε μόνο τις υπόλοιπες
    if existing_count < len(air_final):
        remaining = air_final.iloc[existing_count:].copy()
        print(f"   Υπόλοιπες προς ανέβασμα: {len(remaining)} γραμμές")
    else:
        print("   ✅ Όλες οι γραμμές έχουν ήδη ανεβεί!")
        remaining = pd.DataFrame()
        
except Exception as e:
    print(f"   ⚠️ Δεν μπορώ να ελέγξω, θα ανεβάσω όλες: {e}")
    remaining = air_final.copy()

# ============================================
# 3. ΣΥΝΕΧΙΣΗ UPLOAD (με μικρότερα batches)
# ============================================

if len(remaining) > 0:
    print(f"\n📤 Συνέχιση upload {len(remaining)} γραμμών...")
    
    batch_size = 50
    success = 0
    
    for i in range(0, len(remaining), batch_size):
        batch = remaining.iloc[i:i+batch_size].copy()
        
        # Τελευταίος καθαρισμός για JSON
        batch = batch.where(pd.notnull(batch), None)
        records = batch.to_dict(orient='records')
        
        try:
            supabase.table('air_quality').insert(records).execute()
            success += len(records)
            print(f"   ✅ Ανέβηκαν {success}/{len(remaining)}")
            time.sleep(0.05)
        except Exception as e:
            print(f"   ❌ Σφάλμα στο batch {i}: {e}")
            
            # Δοκίμασε μία-μία τις προβληματικές γραμμές
            print("   🔍 Δοκιμή μία-μία...")
            for j, (idx, row) in enumerate(batch.iterrows()):
                try:
                    record = row.to_dict()
                    # Τελευταίος καθαρισμός
                    for key, val in record.items():
                        if pd.isna(val) or val == np.inf or val == -np.inf:
                            record[key] = 0
                        if isinstance(val, float) and (val != val):  # NaN check
                            record[key] = 0
                    supabase.table('air_quality').insert(record).execute()
                    success += 1
                    if (j + 1) % 10 == 0:
                        print(f"      ✅ {j+1}/{len(batch)}")
                except Exception as e2:
                    print(f"      ❌ Αδύνατη η εγγραφή: {e2}")
                    print(f"         Record: {record}")
            break
    
    print(f"\n📊 Σύνολο που ανέβηκαν σε αυτή τη συνεδρία: {success}")
else:
    print("\n✅ Κανένα νέο δεδομένο για upload")

# ============================================
# 4. ΤΕΛΙΚΟΣ ΕΛΕΓΧΟΣ
# ============================================
print("\n🔍 Τελικός έλεγχος Supabase:")

try:
    final_check = supabase.table('air_quality').select('*', count='exact').execute()
    print(f"   ✅ Αέρας: {final_check.count} γραμμές συνολικά")
    
    water_check = supabase.table('water_quality').select('*', count='exact').execute()
    print(f"   ✅ Νερό: {water_check.count} γραμμές συνολικά")
except Exception as e:
    print(f"   ❌ Σφάλμα: {e}")

✅ Συνδέθηκε στο Supabase

🔄 Καθαρισμός δεδομένων αέρα...
   Πριν: 70128 γραμμές
   Μετά: 70128 γραμμές

🔍 Έλεγχος ήδη ανεβασμένων γραμμών...
   Ήδη ανεβασμένες: 43800 γραμμές
   Υπόλοιπες προς ανέβασμα: 26328 γραμμές

📤 Συνέχιση upload 26328 γραμμών...
   ✅ Ανέβηκαν 50/26328
   ✅ Ανέβηκαν 100/26328
   ✅ Ανέβηκαν 150/26328
   ✅ Ανέβηκαν 200/26328
   ✅ Ανέβηκαν 250/26328
   ✅ Ανέβηκαν 300/26328
   ✅ Ανέβηκαν 350/26328
   ✅ Ανέβηκαν 400/26328
   ✅ Ανέβηκαν 450/26328
   ✅ Ανέβηκαν 500/26328
   ✅ Ανέβηκαν 550/26328
   ✅ Ανέβηκαν 600/26328
   ✅ Ανέβηκαν 650/26328
   ✅ Ανέβηκαν 700/26328
   ✅ Ανέβηκαν 750/26328
   ✅ Ανέβηκαν 800/26328
   ✅ Ανέβηκαν 850/26328
   ✅ Ανέβηκαν 900/26328
   ✅ Ανέβηκαν 950/26328
   ✅ Ανέβηκαν 1000/26328
   ✅ Ανέβηκαν 1050/26328
   ✅ Ανέβηκαν 1100/26328
   ✅ Ανέβηκαν 1150/26328
   ✅ Ανέβηκαν 1200/26328
   ✅ Ανέβηκαν 1250/26328
   ✅ Ανέβηκαν 1300/26328
   ✅ Ανέβηκαν 1350/26328
   ✅ Ανέβηκαν 1400/26328
   ✅ Ανέβηκαν 1450/26328
   ✅ Ανέβηκαν 1500/26328
   ✅ Ανέβηκαν 155

In [25]:
# ============================================
# ΚΕΛΙ 10 – ΕΛΕΓΧΟΣ ΔΕΔΟΜΕΝΩΝ ΣΤΟ SUPABASE
# ============================================

from supabase import create_client
import pandas as pd

# Στοιχεία σύνδεσης
SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Συνδέθηκε στο Supabase\n")

# ============================================
# 1. ΕΛΕΓΧΟΣ ΠΙΝΑΚΑ ΝΕΡΟΥ
# ============================================
print("="*60)
print("📊 ΕΛΕΓΧΟΣ ΠΙΝΑΚΑ ΝΕΡΟΥ (water_quality)")
print("="*60)

try:
    # Συνολικός αριθμός γραμμών
    water_count = supabase.table('water_quality').select('*', count='exact').execute()
    print(f"✅ Συνολικές γραμμές: {water_count.count}")
    
    # Δείγμα δεδομένων (πρώτες 5)
    water_sample = supabase.table('water_quality').select('*').limit(5).execute()
    print(f"\n📋 Δείγμα 5 γραμμών:")
    for i, row in enumerate(water_sample.data):
        print(f"   {i+1}. Έτος:{row.get('year')}, Μήνας:{row.get('month')}, Τοποθεσία:{row.get('location')}, Παράμετρος:{row.get('parameter')}, Τιμή:{row.get('clean_value')}")
    
    # Στατιστικά ανά τοποθεσία
    locations = supabase.table('water_quality').select('location', count='exact').execute()
    print(f"\n📍 Γραμμές ανά τοποθεσία:")
    location_counts = {}
    for row in locations.data:
        loc = row.get('location')
        location_counts[loc] = location_counts.get(loc, 0) + 1
    for loc, count in location_counts.items():
        print(f"   {loc}: {count} γραμμές")
        
except Exception as e:
    print(f"❌ Σφάλμα: {e}")

# ============================================
# 2. ΕΛΕΓΧΟΣ ΠΙΝΑΚΑ ΑΕΡΑ
# ============================================
print("\n" + "="*60)
print("📊 ΕΛΕΓΧΟΣ ΠΙΝΑΚΑ ΑΕΡΑ (air_quality)")
print("="*60)

try:
    # Συνολικός αριθμός γραμμών
    air_count = supabase.table('air_quality').select('*', count='exact').execute()
    print(f"✅ Συνολικές γραμμές: {air_count.count}")
    
    # Δείγμα δεδομένων (πρώτες 5)
    air_sample = supabase.table('air_quality').select('*').limit(5).execute()
    print(f"\n📋 Δείγμα 5 γραμμών:")
    for i, row in enumerate(air_sample.data):
        print(f"   {i+1}. Ημ/νία:{row.get('date')}, Τοποθεσία:{row.get('location')}, CO:{row.get('co')}, NO2:{row.get('no2')}, O3:{row.get('o3')}")
    
    # Στατιστικά ανά έτος
    print(f"\n📅 Γραμμές ανά έτος:")
    years_data = supabase.table('air_quality').select('year', count='exact').execute()
    year_counts = {}
    for row in years_data.data:
        year = row.get('year')
        if year:
            year_counts[year] = year_counts.get(year, 0) + 1
    for year in sorted(year_counts.keys()):
        print(f"   {year}: {year_counts[year]} γραμμές")
    
    # Στατιστικά ανά τοποθεσία
    print(f"\n📍 Γραμμές ανά τοποθεσία:")
    locations_data = supabase.table('air_quality').select('location', count='exact').execute()
    loc_counts = {}
    for row in locations_data.data:
        loc = row.get('location')
        if loc:
            loc_counts[loc] = loc_counts.get(loc, 0) + 1
    for loc, count in loc_counts.items():
        print(f"   {loc}: {count} γραμμές")
    
    # Έλεγχος για κενές τιμές
    print(f"\n🔍 Έλεγχος ποιότητας δεδομένων:")
    
    # Δείγμα για έλεγχο NaN
    null_check = supabase.table('air_quality').select('co,no,no2,so2,o3').limit(100).execute()
    null_counts = {'co':0, 'no':0, 'no2':0, 'so2':0, 'o3':0}
    for row in null_check.data:
        for key in null_counts:
            if row.get(key) is None or row.get(key) == '':
                null_counts[key] += 1
    
    for key, count in null_counts.items():
        print(f"   {key}: {count}/100 γραμμές έχουν κενές τιμές")
    
except Exception as e:
    print(f"❌ Σφάλμα: {e}")

# ============================================
# 3. ΣΥΓΚΡΙΣΗ ΜΕ ΤΑ ΤΟΠΙΚΑ ΑΡΧΕΙΑ
# ============================================
print("\n" + "="*60)
print("📊 ΣΥΓΚΡΙΣΗ ΜΕ ΤΟΠΙΚΑ ΑΡΧΕΙΑ")
print("="*60)

try:
    local_air = pd.read_csv('data/air_final.csv')
    local_water = pd.read_csv('data/water_final.csv')
    
    print(f"✅ Τοπικό αρχείο αέρα: {len(local_air)} γραμμές")
    print(f"✅ Τοπικό αρχείο νερού: {len(local_water)} γραμμές")
    
    if air_count.count == len(local_air):
        print("\n🎉 ΤΑΥΤΙΣΗ! Ο αέρας στο Supabase έχει ΑΚΡΙΒΩΣ τον ίδιο αριθμό γραμμών με το τοπικό αρχείο")
    else:
        diff = len(local_air) - air_count.count
        print(f"\n⚠️ Διαφορά: {diff} γραμμές λείπουν από το Supabase (τοπικό: {len(local_air)}, Supabase: {air_count.count})")
    
    if water_count.count == len(local_water):
        print("🎉 ΤΑΥΤΙΣΗ! Το νερό στο Supabase έχει ΑΚΡΙΒΩΣ τον ίδιο αριθμό γραμμών με το τοπικό αρχείο")
    else:
        diff = len(local_water) - water_count.count
        print(f"⚠️ Διαφορά: {diff} γραμμές λείπουν από το Supabase")
        
except Exception as e:
    print(f"⚠️ Δεν ήταν δυνατή η σύγκριση: {e}")

print("\n" + "="*60)
print("✅ ΕΛΕΓΧΟΣ ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("="*60)

✅ Συνδέθηκε στο Supabase

📊 ΕΛΕΓΧΟΣ ΠΙΝΑΚΑ ΝΕΡΟΥ (water_quality)
✅ Συνολικές γραμμές: 168

📋 Δείγμα 5 γραμμών:
   1. Έτος:2023, Μήνας:Δεκέμβριος, Τοποθεσία:eyosmos, Παράμετρος:Θολότητα NTU, Τιμή:<0.050
   2. Έτος:2023, Μήνας:Δεκέμβριος, Τοποθεσία:eyosmos, Παράμετρος:Χρώμα, Τιμή:<5 
   3. Έτος:2023, Μήνας:Δεκέμβριος, Τοποθεσία:eyosmos, Παράμετρος:Αργίλιο, Τιμή:<25
   4. Έτος:2023, Μήνας:Δεκέμβριος, Τοποθεσία:eyosmos, Παράμετρος:Χλωριούχα, Τιμή:None
   5. Έτος:2023, Μήνας:Δεκέμβριος, Τοποθεσία:eyosmos, Παράμετρος:Αγωγιμότητα, Τιμή:367.0

📍 Γραμμές ανά τοποθεσία:
   eyosmos: 56 γραμμές
   kordelio: 56 γραμμές
   dialogi: 56 γραμμές

📊 ΕΛΕΓΧΟΣ ΠΙΝΑΚΑ ΑΕΡΑ (air_quality)
✅ Συνολικές γραμμές: 70128

📋 Δείγμα 5 γραμμών:
   1. Ημ/νία:2017-01-01, Τοποθεσία:Kordelio-Evosmos, CO:281.36383, NO2:10.874794, O3:41.671654
   2. Ημ/νία:2017-01-01, Τοποθεσία:Kordelio-Evosmos, CO:264.59735, NO2:8.482584, O3:40.620445
   3. Ημ/νία:2017-01-01, Τοποθεσία:Kordelio-Evosmos, CO:257.38742, NO2:6.9105706, O3:42.2

In [26]:
# ============================================
# ΚΕΛΙ 11 – ΚΑΘΑΡΙΣΜΟΣ ΔΙΠΛΟΤΥΠΩΝ ΣΤΟ ΝΕΡΟ
# ============================================

from supabase import create_client
import pandas as pd

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Συνδέθηκε στο Supabase")

# ============================================
# 1. ΔΕΣ ΠΟΙΕΣ ΓΡΑΜΜΕΣ ΕΙΝΑΙ ΔΙΠΛΕΣ
# ============================================
print("\n🔍 Έλεγχος διπλότυπων...")

# Πάρε όλες τις γραμμές
all_water = supabase.table('water_quality').select('*').execute()
df_water = pd.DataFrame(all_water.data)

print(f"Σύνολο γραμμών πριν: {len(df_water)}")

# Έλεγχος διπλότυπων με βάση τα σημαντικά πεδία
duplicate_cols = ['year', 'month', 'location', 'category', 'parameter']
duplicates = df_water.duplicated(subset=duplicate_cols, keep='first')

print(f"Διπλότυπες γραμμές: {duplicates.sum()}")

# ============================================
# 2. ΚΡΑΤΑ ΜΟΝΟ ΤΙΣ ΜΟΝΑΔΙΚΕΣ
# ============================================
df_unique = df_water.drop_duplicates(subset=duplicate_cols, keep='first')
print(f"Μοναδικές γραμμές: {len(df_unique)}")

# ============================================
# 3. ΔΙΑΓΡΑΦΗ ΟΛΩΝ ΚΑΙ ΕΠΑΝΑΠΡΟΣΘΗΚΗ ΜΟΝΑΔΙΚΩΝ
# ============================================
print("\n🔄 Διαγραφή όλων και επαναπροσθήκη...")

try:
    # Διαγραφή όλων
    supabase.table('water_quality').delete().neq('id', 0).execute()
    print("   ✅ Διαγράφηκαν όλες οι γραμμές")
    
    # Προσθήκη μοναδικών
    batch_size = 20
    for i in range(0, len(df_unique), batch_size):
        batch = df_unique.iloc[i:i+batch_size].copy()
        # Αφαίρεση του id πριν την insert
        if 'id' in batch.columns:
            batch = batch.drop('id', axis=1)
        records = batch.to_dict(orient='records')
        supabase.table('water_quality').insert(records).execute()
        print(f"   ✅ Ανέβηκαν {min(i+batch_size, len(df_unique))}/{len(df_unique)}")
    
    print(f"\n✅ Ολοκληρώθηκε! Τώρα το νερό έχει {len(df_unique)} γραμμές")
    
except Exception as e:
    print(f"❌ Σφάλμα: {e}")

# ============================================
# 4. ΤΕΛΙΚΟΣ ΕΛΕΓΧΟΣ
# ============================================
final_check = supabase.table('water_quality').select('*', count='exact').execute()
print(f"\n📊 Τελικός αριθμός γραμμών νερού: {final_check.count}")

✅ Συνδέθηκε στο Supabase

🔍 Έλεγχος διπλότυπων...
Σύνολο γραμμών πριν: 168
Διπλότυπες γραμμές: 84
Μοναδικές γραμμές: 84

🔄 Διαγραφή όλων και επαναπροσθήκη...
   ✅ Διαγράφηκαν όλες οι γραμμές
❌ Σφάλμα: Out of range float values are not JSON compliant

📊 Τελικός αριθμός γραμμών νερού: 0


In [27]:
# ============================================
# ΚΕΛΙ 11 – ΚΑΘΑΡΙΣΜΟΣ & ΕΠΑΝΑΠΡΟΣΘΗΚΗ ΝΕΡΟΥ
# ============================================

from supabase import create_client
import pandas as pd
import numpy as np

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Συνδέθηκε στο Supabase")

# ============================================
# 1. ΦΟΡΤΩΣΗ ΤΟΠΙΚΟΥ ΑΡΧΕΙΟΥ (84 γραμμές)
# ============================================
print("\n📂 Φόρτωση τοπικού αρχείου νερού...")
water_clean = pd.read_csv('data/water_final.csv')
print(f"   Γραμμές: {len(water_clean)}")

# ============================================
# 2. ΚΑΘΑΡΙΣΜΟΣ ΓΙΑ JSON
# ============================================
print("\n🔄 Καθαρισμός δεδομένων...")

# Αντικατάσταση inf, -inf, NaN με None
water_clean = water_clean.replace([np.inf, -np.inf], None)

# Για κάθε στήλη, καθαρισμός
for col in water_clean.columns:
    # Μετατροπή float NaN σε None
    if water_clean[col].dtype == 'float64':
        water_clean[col] = water_clean[col].where(pd.notnull(water_clean[col]), None)
    # Μετατροπή άλλων NaN
    water_clean[col] = water_clean[col].where(pd.notnull(water_clean[col]), None)

# Ειδικός καθαρισμός για τη στήλη clean_value (που μπορεί να έχει <0.050 κλπ)
if 'clean_value' in water_clean.columns:
    water_clean['clean_value'] = water_clean['clean_value'].apply(
        lambda x: str(x) if x is not None and '<' in str(x) else x
    )

print(f"   ✅ Καθαρίστηκαν {len(water_clean)} γραμμές")

# ============================================
# 3. ΔΙΑΓΡΑΦΗ ΥΠΑΡΧΟΝΤΩΝ
# ============================================
print("\n🗑️ Διαγραφή υπαρχόντων δεδομένων...")

try:
    # Διαγραφή όλων (πιο ασφαλής τρόπος)
    supabase.table('water_quality').delete().neq('id', 0).execute()
    print("   ✅ Διαγράφηκαν όλες οι γραμμές")
except Exception as e:
    print(f"   ⚠️ Σφάλμα διαγραφής: {e}")

# ============================================
# 4. ΕΠΑΝΑΠΡΟΣΘΗΚΗ ΜΕ ΜΙΚΡΑ BATCHES
# ============================================
print("\n📤 Επαναπροσθήκη καθαρών δεδομένων...")

batch_size = 10
success = 0

for i in range(0, len(water_clean), batch_size):
    batch = water_clean.iloc[i:i+batch_size].copy()
    
    # Αφαίρεση της στήλης id αν υπάρχει
    if 'id' in batch.columns:
        batch = batch.drop('id', axis=1)
    
    # Καθαρισμός batch
    batch = batch.where(pd.notnull(batch), None)
    records = batch.to_dict(orient='records')
    
    # Τελικός καθαρισμός κάθε record
    clean_records = []
    for record in records:
        clean_record = {}
        for key, value in record.items():
            if value is None:
                clean_record[key] = None
            elif isinstance(value, float) and (np.isnan(value) or np.isinf(value)):
                clean_record[key] = None
            elif isinstance(value, str):
                clean_record[key] = value.strip() if value else None
            else:
                clean_record[key] = value
        clean_records.append(clean_record)
    
    try:
        result = supabase.table('water_quality').insert(clean_records).execute()
        success += len(clean_records)
        print(f"   ✅ Ανέβηκαν {success}/{len(water_clean)}")
    except Exception as e:
        print(f"   ❌ Σφάλμα στο batch {i}: {e}")
        # Δοκίμασε μία-μία
        print("      Δοκιμή μία-μία...")
        for j, record in enumerate(clean_records):
            try:
                supabase.table('water_quality').insert(record).execute()
                success += 1
            except Exception as e2:
                print(f"      ❌ Αποτυχία γραμμής {j}: {e2}")
                print(f"         Record: {record}")

# ============================================
# 5. ΤΕΛΙΚΟΣ ΕΛΕΓΧΟΣ
# ============================================
print("\n🔍 Τελικός έλεγχος...")

try:
    final_check = supabase.table('water_quality').select('*', count='exact').execute()
    print(f"📊 Νερό: {final_check.count} γραμμές")
    
    if final_check.count == 84:
        print("🎉 ΕΠΙΤΥΧΙΑ! Ακριβώς 84 γραμμές ανέβηκαν")
    else:
        print(f"⚠️ Αναμενόταν 84, έχουμε {final_check.count}")
        
    # Δείγμα
    sample = supabase.table('water_quality').select('*').limit(3).execute()
    print("\n📋 Δείγμα:")
    for row in sample.data:
        print(f"   {row.get('location')} - {row.get('parameter')} - {row.get('clean_value')}")
        
except Exception as e:
    print(f"❌ Σφάλμα ελέγχου: {e}")

✅ Συνδέθηκε στο Supabase

📂 Φόρτωση τοπικού αρχείου νερού...
   Γραμμές: 84

🔄 Καθαρισμός δεδομένων...
   ✅ Καθαρίστηκαν 84 γραμμές

🗑️ Διαγραφή υπαρχόντων δεδομένων...
   ✅ Διαγράφηκαν όλες οι γραμμές

📤 Επαναπροσθήκη καθαρών δεδομένων...
   ✅ Ανέβηκαν 10/84
   ✅ Ανέβηκαν 20/84
   ✅ Ανέβηκαν 30/84
   ✅ Ανέβηκαν 40/84
   ✅ Ανέβηκαν 50/84
   ✅ Ανέβηκαν 60/84
   ✅ Ανέβηκαν 70/84
   ✅ Ανέβηκαν 80/84
   ✅ Ανέβηκαν 84/84

🔍 Τελικός έλεγχος...
📊 Νερό: 84 γραμμές
🎉 ΕΠΙΤΥΧΙΑ! Ακριβώς 84 γραμμές ανέβηκαν

📋 Δείγμα:
   eyosmos - Θολότητα NTU - <0.050
   eyosmos - Χρώμα - <5
   eyosmos - Αργίλιο - <25


In [28]:
from supabase import create_client
import pandas as pd

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Δες τα ονόματα στηλών στον πίνακα αέρα
air_sample = supabase.table('air_quality').select('*').limit(1).execute()
print("Στήλες αέρα:", list(air_sample.data[0].keys()) if air_sample.data else "Κανένα δεδομένο")

# Δες τα ονόματα στηλών στο νερό
water_sample = supabase.table('water_quality').select('*').limit(1).execute()
print("Στήλες νερού:", list(water_sample.data[0].keys()) if water_sample.data else "Κανένα δεδομένο")

Στήλες αέρα: ['id', 'timestamp', 'date', 'hour', 'year', 'location', 'co', 'no', 'no2', 'so2', 'o3', 'lon', 'lat', 'created_at']
Στήλες νερού: ['id', 'year', 'month', 'location', 'category', 'parameter', 'unit', 'clean_value', 'limit_value', 'created_at']


In [3]:
from supabase import create_client
SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD"
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Έλεγχος για Κορδελιό 2024
result = supabase.table('air_quality').select('date, no2, o3, co, so2, no').eq('location', 'Kordelio').eq('year', 2024).limit(5).execute()
print(result.data)

[{'date': '2024-01-01', 'no2': 0, 'o3': 0, 'co': 0, 'so2': 0, 'no': 0}, {'date': '2024-01-01', 'no2': 0, 'o3': 0, 'co': 0, 'so2': 0, 'no': 0}, {'date': '2024-01-01', 'no2': 0, 'o3': 0, 'co': 0, 'so2': 0, 'no': 0}, {'date': '2024-01-01', 'no2': 0, 'o3': 0, 'co': 0, 'so2': 0, 'no': 0}, {'date': '2024-01-01', 'no2': 0, 'o3': 0, 'co': 0, 'so2': 0, 'no': 0}]


In [4]:
from supabase import create_client

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Συνδέθηκε\n")
# Έλεγχος για Κορδελιό και NO
for year in [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]:
    result = supabase.table('air_quality')\
        .select('no')\
        .eq('location', 'Kordelio')\
        .eq('year', year)\
        .gt('no', 0)\
        .limit(1)\
        .execute()
    print(f"Kordelio {year}: {'✅ έχει NO' if result.data else '❌ κανένα NO'}")

# Έλεγχος για Εύοσμο
for year in [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]:
    result = supabase.table('air_quality')\
        .select('no2')\
        .eq('location', 'Kordelio-Evosmos')\
        .eq('year', year)\
        .gt('no2', 0)\
        .limit(1)\
        .execute()
    print(f"Evosmos {year}: {'✅ έχει NO₂' if result.data else '❌ κανένα NO₂'}")

✅ Συνδέθηκε

Kordelio 2017: ❌ κανένα NO
Kordelio 2018: ❌ κανένα NO
Kordelio 2019: ❌ κανένα NO
Kordelio 2020: ❌ κανένα NO
Kordelio 2021: ❌ κανένα NO
Kordelio 2022: ❌ κανένα NO
Kordelio 2023: ❌ κανένα NO
Kordelio 2024: ❌ κανένα NO
Evosmos 2017: ✅ έχει NO₂
Evosmos 2018: ✅ έχει NO₂
Evosmos 2019: ✅ έχει NO₂
Evosmos 2020: ❌ κανένα NO₂
Evosmos 2021: ❌ κανένα NO₂
Evosmos 2022: ❌ κανένα NO₂
Evosmos 2023: ❌ κανένα NO₂
Evosmos 2024: ❌ κανένα NO₂


In [5]:
import pandas as pd
import os

# Φόρτωσε ένα αρχείο αέρα για Κορδελιό (π.χ. 2017)
air_folder = 'data/air/'
files = [f for f in os.listdir(air_folder) if 'kordeliou' in f.lower() and 'euosmou' not in f.lower()]

if files:
    df_test = pd.read_csv(os.path.join(air_folder, files[0]))
    print("Στήλες:", df_test.columns.tolist())
    print("\nΠρώτες 3 γραμμές (στήλη NO):")
    print(df_test[['time', 'no']].head() if 'no' in df_test.columns else "Δεν υπάρχει στήλη 'no'")
else:
    print("Δεν βρέθηκαν αρχεία για Κορδελιό")

FileNotFoundError: [WinError 3] Το σύστημα δεν είναι σε θέση να εντοπίσει την καθορισμένη διαδρομή δίσκου: 'data/air/'

In [29]:
from supabase import create_client

SUPABASE_URL = "https://hjfbcknlnvgwsfqhfxwx.supabase.co"  # Αντικατέστησε
SUPABASE_KEY = "sb_publishable_YOlkfQYde0xMkRHe5QOASA_iHLiOBVD"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Συνδέθηκε\n")

print("="*60)
print("ΑΚΡΙΒΗ ΣΤΑΤΙΣΤΙΚΑ ΔΕΔΟΜΕΝΩΝ ΑΕΡΑ")
print("="*60)

# Συνολικές γραμμές
total = supabase.table('air_quality').select('*', count='exact').execute()
print(f"\n📊 Σύνολο γραμμών στον πίνακα: {total.count}")

# Αριθμός γραμμών ανά ρύπο με τιμή > 0
pollutants = ['no2', 'o3', 'co', 'so2', 'no']

for p in pollutants:
    try:
        result = supabase.table('air_quality').select(p, count='exact').gt(p, 0).execute()
        print(f"{p.upper()}: {result.count} γραμμές με τιμή > 0")
    except Exception as e:
        print(f"{p.upper()}: Σφάλμα - {e}")

# Έλεγχος για την πιο πρόσφατη μη-μηδενική τιμή
print("\n" + "="*60)
print("ΤΕΛΕΥΤΑΙΕΣ ΜΗ-ΜΗΔΕΝΙΚΕΣ ΤΙΜΕΣ")
print("="*60)

for p in pollutants:
    try:
        result = supabase.table('air_quality').select('date', p).gt(p, 0).order('date', desc=True).limit(1).execute()
        if result.data:
            print(f"{p.upper()}: {result.data[0][p]} στις {result.data[0]['date']}")
        else:
            print(f"{p.upper()}: ΔΕΝ ΥΠΑΡΧΕΙ ΚΑΜΙΑ ΤΙΜΗ > 0")
    except Exception as e:
        print(f"{p.upper()}: Σφάλμα - {e}")

print("\n" + "="*60)
print("ΕΛΕΓΧΟΣ ΝΕΡΟΥ")
print("="*60)

water_total = supabase.table('water_quality').select('*', count='exact').execute()
print(f"Σύνολο γραμμών νερού: {water_total.count}")

# Ανά τοποθεσία
for loc in ['eyosmos', 'kordelio', 'dialogi']:
    count = supabase.table('water_quality').select('*', count='exact').eq('location', loc).execute()
    print(f"{loc}: {count.count} γραμμές")

✅ Συνδέθηκε

ΑΚΡΙΒΗ ΣΤΑΤΙΣΤΙΚΑ ΔΕΔΟΜΕΝΩΝ ΑΕΡΑ

📊 Σύνολο γραμμών στον πίνακα: 70128
NO2: 26280 γραμμές με τιμή > 0
O3: 26280 γραμμές με τιμή > 0
CO: 26280 γραμμές με τιμή > 0
SO2: 26272 γραμμές με τιμή > 0
NO: 17520 γραμμές με τιμή > 0

ΤΕΛΕΥΤΑΙΕΣ ΜΗ-ΜΗΔΕΝΙΚΕΣ ΤΙΜΕΣ
NO2: 4.408203 στις 2019-12-31
O3: 38.53125 στις 2019-12-31
CO: 199 στις 2019-12-31
SO2: 4.875 στις 2019-12-31
NO: 0.0059280396 στις 2019-12-31

ΕΛΕΓΧΟΣ ΝΕΡΟΥ
Σύνολο γραμμών νερού: 84
eyosmos: 28 γραμμές
kordelio: 28 γραμμές
dialogi: 28 γραμμές
